# Customer Churn — Evaluation Visuals

Run the training pipeline first. This notebook inspects held-out probabilities, threshold behaviour, and risk bands.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ARTIFACTS = Path("../artifacts")
predictions = pd.read_csv(ARTIFACTS / "test_predictions.csv")
metrics = json.loads((ARTIFACTS / "metrics.json").read_text())
metrics


In [ ]:
figure, axis = plt.subplots(figsize=(9, 5))
for label, group in predictions.groupby("actual"):
    axis.hist(group["churn_probability"], bins=20, alpha=0.55, label=f"Actual {label}")
axis.axvline(metrics["decision_threshold"], color="#DC2626", linestyle="--", label="Selected threshold")
axis.set(title="Held-out Churn Probability Distribution", xlabel="Churn probability", ylabel="Customers")
axis.legend()
figure.tight_layout()


In [ ]:
predictions["risk_band"] = pd.qcut(predictions["churn_probability"], 10, duplicates="drop")
bands = predictions.groupby("risk_band", observed=True).agg(
    Customers=("actual", "size"),
    ObservedChurnRate=("actual", "mean"),
    AverageScore=("churn_probability", "mean"),
).reset_index()
figure, axis = plt.subplots(figsize=(10, 5))
axis.plot(range(1, len(bands) + 1), bands["ObservedChurnRate"], marker="o", label="Observed")
axis.plot(range(1, len(bands) + 1), bands["AverageScore"], marker="o", label="Predicted")
axis.set(title="Churn Risk Deciles", xlabel="Risk decile", ylabel="Rate")
axis.legend()
axis.grid(alpha=0.25)
figure.tight_layout()
